In [17]:
from transformers import pipeline
from PIL import Image
import requests

In [15]:
#Sentiment analysis pipeline
text_classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

text_tests = [
    "I absolutely love this phone.",
    "This is the worst movie I have ever seen.",
    "The food was okay, not amazing but not terrible."
]

for text in text_tests:
    result = text_classifier(text)
    print(f"\nInput: {text}")
    print("Result:", result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Input: I absolutely love this phone.
Result: [{'label': 'POSITIVE', 'score': 0.999860405921936}]

Input: This is the worst movie I have ever seen.
Result: [{'label': 'NEGATIVE', 'score': 0.9997548460960388}]

Input: The food was okay, not amazing but not terrible.
Result: [{'label': 'POSITIVE', 'score': 0.9936789870262146}]


Observations
- Positive sentences are classified well.
- Negative sentences are usually very accurate.
- Neutral sentences can confuse the model slightly.

In [16]:
#Named Entity Recognition (NER)
token_classifier = pipeline(
    "token-classification",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"
)

token_tests = [
    "Elon Musk founded SpaceX in the United States.",
    "Apple released a new iPhone in California.",
    "Lionel Messi plays football for Argentina."
]

for text in token_tests:
    result = token_classifier(text)

    print(f"\nInput: {text}")
    for entity in result:
        print(entity)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Input: Elon Musk founded SpaceX in the United States.
{'entity_group': 'PER', 'score': np.float32(0.99837005), 'word': 'Elon Musk', 'start': 0, 'end': 9}
{'entity_group': 'ORG', 'score': np.float32(0.9985827), 'word': 'SpaceX', 'start': 18, 'end': 24}
{'entity_group': 'LOC', 'score': np.float32(0.9996748), 'word': 'United States', 'start': 32, 'end': 45}

Input: Apple released a new iPhone in California.
{'entity_group': 'ORG', 'score': np.float32(0.9975387), 'word': 'Apple', 'start': 0, 'end': 5}
{'entity_group': 'MISC', 'score': np.float32(0.99374384), 'word': 'iPhone', 'start': 21, 'end': 27}
{'entity_group': 'LOC', 'score': np.float32(0.9997701), 'word': 'California', 'start': 31, 'end': 41}

Input: Lionel Messi plays football for Argentina.
{'entity_group': 'PER', 'score': np.float32(0.99923307), 'word': 'Lionel Messi', 'start': 0, 'end': 12}
{'entity_group': 'LOC', 'score': np.float32(0.9997986), 'word': 'Argentina', 'start': 32, 'end': 41}


Observations
- People names are recognized very well.
- Locations and organizations are usually correct.
- Some entities may occasionally be split incorrectly.

In [ ]:
#Load multimodal pipeline
multimodal_pipeline = pipeline(
    "image-text-to-text",
    model="Salesforce/blip-image-captioning-base"
)

image_tests = [
    {
        "url": "https://images.unsplash.com/photo-1518791841217-8f162f1e1131",
        "prompt": "Describe this image:"
    },
    {
        "url": "https://images.unsplash.com/photo-1503023345310-bd7c1de61c7d",
        "prompt": "What is happening in this picture?"
    },
    {
        "url": "https://images.unsplash.com/photo-1495567720989-cebdbdd97913",
        "prompt": "Write a short caption for this image."
    }
]

for test in image_tests:

    image = Image.open(requests.get(test["url"], stream=True).raw)

    result = multimodal_pipeline(
        images=image,
        text=test["prompt"]
    )

    print(f"\nImage URL: {test['url']}")
    print(f"Prompt: {test['prompt']}")
    print("Result:", result[0]['generated_text'])


IMAGE TEXT TO TEXT


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Image URL: https://images.unsplash.com/photo-1518791841217-8f162f1e1131
Prompt: Describe this image:
Result: Describe this image: cat


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Image URL: https://images.unsplash.com/photo-1503023345310-bd7c1de61c7d
Prompt: What is happening in this picture?
Result: What is happening in this picture?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Image URL: https://images.unsplash.com/photo-1495567720989-cebdbdd97913
Prompt: Write a short caption for this image.
Result: Write a short caption for this image. sunset

Observations:
- The model generates captions from both image and text input.
- Simple prompts work best.
- Results are usually accurate for common objects and scenes.


Observations
- The model generates captions from both image and text input.
- Simple prompts work best.
- Results are usually accurate for common objects and scenes.

FINAL THOUGHTS

- Text classification is very accurate for clear emotions.
- Token classification works well for names, places, and organizations.
- The multimodal BLIP model produces human-like image captions.

Overall, pretrained transformer pipelines are easy to use and perform very well
without needing extra training.